# Google TTS demo

**This notebook does not build datasets.** It exercises Google Cloud Text-to-Speech voices and languages defined in `src/constants.py` (`TTS_CONFIGS`) before you run long synthesis jobs in the builder notebooks.

## Purpose

Helps pick TTS language codes and hear male/female output quality. Useful when adding a new locale or checking API credentials—not for filtering, sampling, or writing Hub parquets.

This notebook:

1. Instantiates `TTS` with your `.env` Google credentials.
2. Iterates configured languages and genders from `TTS_CONFIGS`.
3. Synthesizes sample phrases and plays them inline.

Use it to sanity-check TTS settings; run `voice_bench.ipynb` or `infinity_instruct.ipynb` for full dataset builds.


# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from src.setup import configure_notebook_environment

configure_notebook_environment(set_plot_style=False)

PosixPath('/Users/pawelp/Desktop/education/pw/bachelor/MLLM-Shap')

In [ ]:
from pprint import pprint

from src.google_tts_demo import (
    build_configured_samples_dataframe,
    display_all_tts_samples,
    filter_voices_for_demo,
    list_voices_dataframe,
)
from src.nlp import TTS

For our research we'll use british english.

In [4]:
LANGUAGE_CODES: tuple[str, ...] = ("en-GB", "es-ES", "fr-FR")

SAMPLES: dict[str, str] = {
    "en": (
        "Provide academic quotes that specifically talk about "
        "the relevance of the social component in the learning process.?"
    ),
    "es": (
        "Proporciona citas académicas que hablen específicamente "
        "sobre la relevancia del componente social en el proceso de aprendizaje."
    ),
    "fr": (
        "Fournissez des citations académiques qui parlent spécifiquement "
        "de la pertinence de la composante sociale dans le processus d'apprentissage"
    ),
}

tts = TTS()

# TTS Demo

In [5]:
TTS.display_audio(
    await tts.synthesize_text(
        text="Hello, this is a demo of Google Text-to-Speech.",
        language_code="en-GB",
        voice_name="en-GB-Wavenet-D",
        gender=1,
    )
)

In [6]:
voices_df = await list_voices_dataframe(tts)
voices_df.head(3)

,name,language_codes,gender
0,Achernar,[en-US],FEMALE
1,Achird,[en-US],MALE
2,Algenib,[en-US],MALE


# Voices analysis

Let's assemble all available voices into a pandas data frame.

In [7]:
pprint(sorted(voices_df.explode("language_codes")["language_codes"].unique()))

['af-ZA',
 'am-ET',
 'ar-XA',
 'bg-BG',
 'bn-IN',
 'ca-ES',
 'cmn-CN',
 'cmn-TW',
 'cs-CZ',
 'da-DK',
 'de-DE',
 'el-GR',
 'en-AU',
 'en-GB',
 'en-IN',
 'en-US',
 'es-ES',
 'es-US',
 'et-EE',
 'eu-ES',
 'fi-FI',
 'fil-PH',
 'fr-CA',
 'fr-FR',
 'gl-ES',
 'gu-IN',
 'he-IL',
 'hi-IN',
 'hr-HR',
 'hu-HU',
 'id-ID',
 'is-IS',
 'it-IT',
 'ja-JP',
 'kn-IN',
 'ko-KR',
 'lt-LT',
 'lv-LV',
 'ml-IN',
 'mr-IN',
 'ms-MY',
 'nb-NO',
 'nl-BE',
 'nl-NL',
 'pa-IN',
 'pl-PL',
 'pt-BR',
 'pt-PT',
 'ro-RO',
 'ru-RU',
 'sk-SK',
 'sl-SI',
 'sr-RS',
 'sv-SE',
 'sw-KE',
 'ta-IN',
 'te-IN',
 'th-TH',
 'tr-TR',
 'uk-UA',
 'ur-IN',
 'vi-VN',
 'yue-HK']


Variety of languages are supported. Note that there are different dialects supported withing one language, ex. for english.

In [8]:
voices_df = filter_voices_for_demo(voices_df, LANGUAGE_CODES)
voices_df.head(3)

,name,language_codes,gender,model_name
0,en-GB-Chirp-HD-F,en-GB,FEMALE,F
1,en-GB-Chirp-HD-O,en-GB,FEMALE,O
2,en-GB-Chirp3-HD-Achernar,en-GB,FEMALE,Achernar


In [9]:
voices_df[["gender", "model_name"]].drop_duplicates().sort_values(
    by=["gender", "model_name"]
).reset_index(drop=True)

,gender,model_name
0,FEMALE,Achernar
1,FEMALE,Aoede
2,FEMALE,Autonoe
3,FEMALE,Callirrhoe
4,FEMALE,Despina
5,FEMALE,Erinome
6,FEMALE,F
7,FEMALE,Gacrux
8,FEMALE,Kore
9,FEMALE,Laomedeia


We'll limit the search for HD models only, as they limit our scope to Chrisp 3: HD voices, that are, according to [Google TTS Docs](https://cloud.google.com/text-to-speech/pricing?hl=en), the latest TTS models as of 2025-10-11.

In [10]:
voices_df[voices_df["model_name"].isin(["F", "D"])][
    ["language_codes", "gender", "name"]
].drop_duplicates().sort_values(by=["language_codes", "gender"]).reset_index(
    drop=True
).rename(columns={"language_codes": "language_code"})

,language_code,gender,name
0,en-GB,FEMALE,en-GB-Chirp-HD-F
1,en-GB,MALE,en-GB-Chirp-HD-D
2,es-ES,FEMALE,es-ES-Chirp-HD-F
3,es-ES,MALE,es-ES-Chirp-HD-D
4,fr-FR,FEMALE,fr-FR-Chirp-HD-F
5,fr-FR,MALE,fr-FR-Chirp-HD-D


Limit models list to only those available in all languages.

In [11]:
del voices_df

There are still many models to choose from. Official demo by Google uses F for Female and D for male, and that's what we'll use as well.

In [12]:
sample_df = await build_configured_samples_dataframe(tts, SAMPLES)
sample_df.head(3)

,language_code,gender,voice_name,audio
0,fr-FR,1,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
1,fr-FR,2,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
2,en-GB,1,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...


In [13]:
display_all_tts_samples(sample_df, tts=tts)

Language: fr-FR, Gender: 1, Voice: None


Language: fr-FR, Gender: 2, Voice: None


Language: en-GB, Gender: 1, Voice: None


Language: en-GB, Gender: 2, Voice: None


Language: es-ES, Gender: 1, Voice: None


Language: es-ES, Gender: 2, Voice: None
